In [ ]:
# ================== MAPA DESDE MATRIZ PRECALCULADA ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
from folium.plugins import MarkerCluster, HeatMap
from shapely.geometry import MultiPoint, Point
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import ScatterChart, Reference, Series
import warnings
warnings.filterwarnings('ignore')

# Buscar archivos Excel en la carpeta de inputs
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Input_vol_import'
archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))

if archivos_excel:
    archivo_matriz = archivos_excel[0]  # Tomar el primer archivo encontrado
    print(f"Archivo seleccionado: {os.path.basename(archivo_matriz)}")
else:
    print("No se encontraron archivos Excel en la carpeta de inputs")
    archivo_matriz = None

output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

# Leer y explorar el archivo Excel
if archivo_matriz:
    try:
        # Obtener todas las hojas disponibles
        excel_file = pd.ExcelFile(archivo_matriz)
        hojas_disponibles = excel_file.sheet_names
        print(f"📋 Hojas disponibles en el archivo:")
        for i, hoja in enumerate(hojas_disponibles, 1):
            print(f"   {i}. {hoja}")
        
        # Buscar hoja que contenga datos de recorridos (buscar palabras clave)
        hoja_seleccionada = None
        palabras_clave = ['recorrido', 'ruta', 'cliente', 'met', 'datos']
        
        for hoja in hojas_disponibles:
            hoja_lower = hoja.lower()
            if any(palabra in hoja_lower for palabra in palabras_clave):
                hoja_seleccionada = hoja
                break
        
        # Si no se encuentra por palabras clave, usar la primera hoja
        if not hoja_seleccionada:
            hoja_seleccionada = hojas_disponibles[0]
        
        print(f"✅ Hoja seleccionada automáticamente: '{hoja_seleccionada}'")
        
        # Leer la hoja seleccionada
        df_recorridos = pd.read_excel(archivo_matriz, sheet_name=hoja_seleccionada)
        print(f"📊 Datos cargados: {df_recorridos.shape[0]} filas, {df_recorridos.shape[1]} columnas")
        
        # Mostrar las primeras columnas para verificar
        print(f"🔍 Columnas encontradas:")
        for i, col in enumerate(df_recorridos.columns[:10], 1):  # Mostrar solo las primeras 10
            print(f"   {i}. {col}")
        if len(df_recorridos.columns) > 10:
            print(f"   ... y {len(df_recorridos.columns) - 10} columnas más")
            
    except Exception as e:
        print(f"❌ Error al leer el archivo Excel: {e}")
        df_recorridos = None
else:
    print("❌ No se encontró archivo Excel para procesar")
    df_recorridos = None

# Verificar que el DataFrame fue cargado correctamente antes de continuar
if df_recorridos is not None and not df_recorridos.empty:
    # Mapeo automático de columnas
    def encontrar_columna(palabras_clave, columnas_disponibles):
        for col in columnas_disponibles:
            col_clean = col.lower().replace('🏠', '').replace('🛣️', '').replace('🔢', '').replace('🆔', '').replace('🌍', '').replace('📚', '').replace('📏', '').strip()
            for palabra in palabras_clave:
                if palabra.lower() in col_clean:
                    return col
        return None
    
    # Buscar columnas automáticamente
    col_met = encontrar_columna(['met'], df_recorridos.columns) or 'MET'
    col_ruta = encontrar_columna(['ruta'], df_recorridos.columns) or 'Ruta'
    col_secuencia = encontrar_columna(['secuencia'], df_recorridos.columns) or 'Secuencia'
    col_tipo = encontrar_columna(['tipo'], df_recorridos.columns) or 'Tipo'
    col_codigo = encontrar_columna(['codigo'], df_recorridos.columns) or '🆔 Codigo'
    col_longitud = encontrar_columna(['longitud'], df_recorridos.columns) or '🌍 Longitud'
    col_latitud = encontrar_columna(['latitud'], df_recorridos.columns) or '🌍 Latitud'
    col_nombre = encontrar_columna(['nombre'], df_recorridos.columns) or '📚 Nombre'
    col_distancia_sig = encontrar_columna(['distancia al siguiente'], df_recorridos.columns) or '📏 Distancia_al_siguiente_km'
    col_volumen = encontrar_columna(['volumen promedio diario', 'volumen'], df_recorridos.columns) or 'Volumen promedio diario'
    col_importe = encontrar_columna(['importe promedio diario', 'importe'], df_recorridos.columns) or 'Importe promedio diario'
    
    print(f"🔍 Mapeo automático de columnas:")
    print(f"   MET: {col_met}")
    print(f"   Ruta: {col_ruta}")
    print(f"   Secuencia: {col_secuencia}")
    print(f"   Tipo: {col_tipo}")
    print(f"   Codigo: {col_codigo}")
    print(f"   Longitud: {col_longitud}")
    print(f"   Latitud: {col_latitud}")
    print(f"   Nombre: {col_nombre}")
    
    # Verificar qué columnas existen realmente
    columnas_requeridas = [col_met, col_ruta, col_secuencia, col_tipo, col_codigo, 
                          col_longitud, col_latitud, col_nombre]
    columnas_opcionales = [col_distancia_sig, col_volumen, col_importe]
    
    print(f"🔍 Verificando columnas requeridas:")
    columnas_faltantes = []
    for col in columnas_requeridas:
        if col in df_recorridos.columns:
            print(f"   ✅ {col}")
        else:
            print(f"   ❌ {col} - NO ENCONTRADA")
            columnas_faltantes.append(col)
    
    print(f"🔍 Verificando columnas opcionales:")
    for col in columnas_opcionales:
        if col in df_recorridos.columns:
            print(f"   ✅ {col}")
        else:
            print(f"   ⚠️ {col} - No encontrada (se usarán valores por defecto)")
    
    if columnas_faltantes:
        print(f"\n❌ FALTAN COLUMNAS CRÍTICAS: {columnas_faltantes}")
        print("🔍 Columnas disponibles en el archivo:")
        for col in df_recorridos.columns:
            print(f"   - {col}")
        df_recorridos = None  # Marcar como inválido
    else:
        print(f"\n✅ Todas las columnas críticas están disponibles")
        
        # Función para normalizar y limpiar datos numéricos
        def limpiar_numerico(valor):
            if pd.isna(valor) or valor == '' or valor == 0:
                return 0
            try:
                return float(valor)
            except:
                return 0

        # Limpiar y normalizar columnas numéricas si existen
        if col_volumen in df_recorridos.columns:
            df_recorridos[col_volumen] = df_recorridos[col_volumen].apply(limpiar_numerico)
            print(f"✅ Columna {col_volumen} normalizada")
        else:
            # Crear columna con valores por defecto
            df_recorridos[col_volumen] = 1.0
            print(f"⚠️ Columna {col_volumen} creada con valores por defecto")
            
        if col_importe in df_recorridos.columns:
            df_recorridos[col_importe] = df_recorridos[col_importe].apply(limpiar_numerico)
            print(f"✅ Columna {col_importe} normalizada")
        else:
            # Crear columna con valores por defecto
            df_recorridos[col_importe] = 100.0
            print(f"⚠️ Columna {col_importe} creada con valores por defecto")
            
        if col_distancia_sig not in df_recorridos.columns:
            df_recorridos[col_distancia_sig] = 0.0
            print(f"⚠️ Columna {col_distancia_sig} creada con valores por defecto")

        # Función para obtener el tamaño del marcador basado en volumen (MEJORADA)
        def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=8, tamano_max=40):
            if vol_max == vol_min or volumen == 0:
                return tamano_min
            # Usar escala logarítmica para mejor diferenciación visual
            if volumen <= 0:
                return tamano_min
            log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
            log_min = np.log1p(vol_min)
            log_max = np.log1p(vol_max)
            if log_max == log_min:
                return tamano_min
            proporcion = (log_vol - log_min) / (log_max - log_min)
            return int(tamano_min + proporcion * (tamano_max - tamano_min))

        # Función para obtener color basado en importe (MEJORADA - escala amarillo-naranja-rojo)
        def obtener_color_importe(importe, imp_min, imp_max):
            if imp_max == imp_min or importe == 0:
                return '#808080'  # Gris para valores sin importe
            proporcion = (importe - imp_min) / (imp_max - imp_min)
            # Escala mejorada: amarillo (bajo) → naranja (medio) → rojo (alto)
            if proporcion <= 0.5:
                # De amarillo a naranja
                rojo = int(255 * (0.8 + proporcion * 0.4))  # 204 a 255
                verde = int(255 * (1.0 - proporcion * 0.4))  # 255 a 204
                azul = 0
            else:
                # De naranja a rojo
                rojo = 255
                verde = int(255 * (1.6 - proporcion * 1.6))  # 204 a 0
                azul = 0
            return f'#{rojo:02x}{verde:02x}{azul:02x}'

        # NOTA: Los rangos ahora se calculan dinámicamente dentro de generar_mapa() 
        # basándose en la selección de METs y rutas del usuario
        # 
        # NOTA: Los rangos ahora se calculan dinámicamente dentro de generar_mapa() 
        # basándose en la selección de METs y rutas del usuario
        # 
        # # Calcular rangos de volumen e importe para clientes
        # clientes_df = df_recorridos[df_recorridos[col_tipo] == 'Cliente']
        # if not clientes_df.empty:
        #     vol_min = clientes_df[col_volumen].min()
        #     vol_max = clientes_df[col_volumen].max()
        #     imp_min = clientes_df[col_importe].min()
        #     imp_max = clientes_df[col_importe].max()
        #     
        #     print(f"📊 Rangos de datos para clientes:")
        #     print(f"   Volumen: {vol_min:,.2f} - {vol_max:,.2f}")
        #     print(f"   Importe: ${imp_min:,.2f} - ${imp_max:,.2f}")
        #     print(f"   Total clientes: {len(clientes_df)}")
        # else:
        #     vol_min = vol_max = imp_min = imp_max = 0
        #     print("⚠️ No se encontraron registros de clientes")

        # Inicializar variables de rangos (serán calculadas dinámicamente)
        vol_min = vol_max = imp_min = imp_max = 0

        mets = sorted(df_recorridos[col_met].dropna().unique())
        rutas = sorted(df_recorridos[col_ruta].dropna().unique())
        
        print(f"📍 METs encontrados: {len(mets)}")
        print(f"🛣️ Rutas encontradas: {len(rutas)}")
else:
    print("❌ No se puede continuar: DataFrame no válido o vacío")
    # Crear variables vacías para evitar errores
    mets = []
    rutas = []
    vol_min = vol_max = imp_min = imp_max = 0

# Controles interactivos MEJORADOS
met_selector = widgets.SelectMultiple(
    options=mets, 
    value=tuple(mets), 
    description='📍 Selecciona MET(s):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='50%')
)

ruta_selector = widgets.SelectMultiple(
    options=rutas, 
    value=tuple(rutas), 
    description='🛣️ Selecciona Ruta(s):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='50%')
)

todos_clientes_checkbox = widgets.Checkbox(
    value=True, 
    description='Procesar todos los clientes', 
    indent=False
)

todas_rutas_checkbox = widgets.Checkbox(
    value=True, 
    description='Seleccionar todas las rutas', 
    indent=False
)

clientes_a_procesar = widgets.IntText(
    value=10, 
    description='Cantidad de clientes:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='40%')
)

# NUEVOS CONTROLES PARA VISUALIZACIÓN
tipo_visualizacion = widgets.Dropdown(
    options=[
        ('🎯 Marcadores inteligentes (Recomendado)', 'marcadores_smart'),
        ('📍 Marcadores básicos', 'marcadores'),
        ('🔥 Solo mapas de calor', 'solo_heatmap'),
        ('🎨 Completo (Marcadores + Calor)', 'completo')
    ],
    value='marcadores_smart',
    description='🎨 Visualización:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

# Casillas de mapas de calor removidas - ahora se controlan desde el dropdown principal

escala_marcadores = widgets.Dropdown(
    options=[
        ('🔹 Pequeños (6-25px)', 'small'),
        ('🔸 Medianos (8-35px)', 'medium'),
        ('🔶 Grandes (10-50px)', 'large')
    ],
    value='medium',
    description='📏 Tamaño marcadores:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

generar_btn = widgets.Button(
    description='🗺️ Generar Mapa de Análisis', 
    button_style='success',
    layout=widgets.Layout(width='300px', height='40px')
)

analisis_excel_btn = widgets.Button(
    description='📊 Generar Análisis Excel', 
    button_style='info',
    layout=widgets.Layout(width='300px', height='40px')
)

output_map = widgets.Output()
output_excel = widgets.Output()

# Actualiza rutas disponibles según MET seleccionado
def actualizar_rutas(*args):
    mets_seleccionados = list(met_selector.value)
    if mets_seleccionados:
        rutas_filtradas = sorted(df_recorridos[df_recorridos[col_met].isin(mets_seleccionados)][col_ruta].dropna().unique())
    else:
        rutas_filtradas = sorted(df_recorridos[col_ruta].dropna().unique())
    ruta_selector.options = rutas_filtradas
    rutas_validas = [r for r in ruta_selector.value if r in rutas_filtradas]
    if rutas_validas:
        ruta_selector.value = tuple(rutas_validas)
    elif rutas_filtradas:
        ruta_selector.value = (rutas_filtradas[0],)
    else:
        ruta_selector.value = ()

# Función para controlar selección de todas las rutas
def controlar_todas_rutas(*args):
    if todas_rutas_checkbox.value:
        # Seleccionar todas las rutas disponibles
        ruta_selector.value = tuple(ruta_selector.options)
    else:
        # Deseleccionar todas las rutas
        ruta_selector.value = ()

# Función para actualizar el checkbox cuando se cambian las rutas manualmente
def actualizar_checkbox_rutas(*args):
    # Si todas las rutas están seleccionadas, marcar el checkbox
    todas_rutas_checkbox.value = len(ruta_selector.value) == len(ruta_selector.options) and len(ruta_selector.options) > 0

met_selector.observe(actualizar_rutas, names='value')
todas_rutas_checkbox.observe(controlar_todas_rutas, names='value')
ruta_selector.observe(actualizar_checkbox_rutas, names='value')
actualizar_rutas()

# Paleta de colores para MET General
colores_met_general = ['#FF6B00', '#00529B', '#FFA500', '#8BC34A', '#E91E63', '#00BCD4', '#FF9800', '#9C27B0', '#607D8B', '#CDDC39', '#795548', '#3F51B5', '#F44336', '#009688', '#C0CA33', '#7B1FA2', '#D32F2F', '#388E3C', '#1976D2', '#FFA07A']

def generar_mapa(b):
    with output_map:
        clear_output()
        
        # Verificar que los datos estén disponibles
        if df_recorridos is None or df_recorridos.empty:
            print('❌ No hay datos válidos para generar el mapa.')
            print('Por favor, verifica que el archivo Excel contenga las columnas necesarias.')
            return
            
        mets_seleccionados = list(met_selector.value)
        rutas_seleccionadas = list(ruta_selector.value)
        if not mets_seleccionados:
            print('Selecciona al menos un MET para generar el mapa.')
            return
        if not rutas_seleccionadas:
            print('Selecciona al menos una ruta para generar el mapa.')
            return

        print(f"🚀 Generando mapa con visualización '{tipo_visualizacion.value}'...")
        
        # CALCULAR RANGOS DINÁMICAMENTE basándose en la selección del usuario
        clientes_seleccionados = df_recorridos[
            (df_recorridos[col_met].isin(mets_seleccionados)) & 
            (df_recorridos[col_ruta].isin(rutas_seleccionadas)) & 
            (df_recorridos[col_tipo] == 'Cliente')
        ]
        
        if not clientes_seleccionados.empty:
            vol_min = clientes_seleccionados[col_volumen].min()
            vol_max = clientes_seleccionados[col_volumen].max()
            imp_min = clientes_seleccionados[col_importe].min()
            imp_max = clientes_seleccionados[col_importe].max()
            
            print(f"📊 Rangos dinámicos para la selección actual:")
            print(f"   📦 Volumen: {vol_min:,.3f} - {vol_max:,.3f}")
            print(f"   💰 Importe: ${imp_min:,.2f} - ${imp_max:,.2f}")
            print(f"   👥 Clientes en selección: {len(clientes_seleccionados)}")
        else:
            vol_min = vol_max = imp_min = imp_max = 0
            print("⚠️ No se encontraron clientes en la selección actual")
        
        met_row = df_recorridos[df_recorridos[col_met] == mets_seleccionados[0]].iloc[0]
        mapa = folium.Map(location=[met_row[col_latitud], met_row[col_longitud]], zoom_start=10, tiles='OpenStreetMap')
        
        # Obtener configuración de tamaños
        tamanos_config = {
            'small': (6, 25),
            'medium': (8, 35),
            'large': (10, 50)
        }
        tamano_min, tamano_max = tamanos_config[escala_marcadores.value]
        
        # Preparar datos para mapas de calor
        clientes_todos = df_recorridos[
            (df_recorridos[col_met].isin(mets_seleccionados)) & 
            (df_recorridos[col_ruta].isin(rutas_seleccionadas)) & 
            (df_recorridos[col_tipo] == 'Cliente')
        ]
        
        print(f"📊 Analizando {len(clientes_todos)} clientes...")
        
        # Datos para mapas de calor
        heat_data_volumen = []
        heat_data_importe = []
        for _, row in clientes_todos.iterrows():
            lat, lon = row[col_latitud], row[col_longitud]
            vol = row[col_volumen] if col_volumen in df_recorridos.columns and not pd.isna(row[col_volumen]) else 0
            imp = row[col_importe] if col_importe in df_recorridos.columns and not pd.isna(row[col_importe]) else 0
            if vol > 0:
                heat_data_volumen.append([lat, lon, vol])
            if imp > 0:
                heat_data_importe.append([lat, lon, imp])
        
        # Añadir mapas de calor si se solicitan
        if tipo_visualizacion.value in ['solo_heatmap', 'completo']:
            if heat_data_volumen:
                HeatMap(
                    heat_data_volumen, 
                    name='🔥 Mapa de Calor - Volumen', 
                    radius=25, 
                    blur=15, 
                    max_zoom=1,
                    gradient={0.2: 'yellow', 0.4: 'orange', 0.6: 'darkorange', 0.8: 'red', 1: 'darkred'},
                    show=False
                ).add_to(mapa)
                print("✅ Mapa de calor por volumen añadido")
        
        if tipo_visualizacion.value in ['solo_heatmap', 'completo']:
            if heat_data_importe:
                HeatMap(
                    heat_data_importe, 
                    name='💰 Mapa de Calor - Importe', 
                    radius=25, 
                    blur=15, 
                    max_zoom=1,
                    gradient={0.2: 'yellow', 0.4: 'orange', 0.6: 'darkorange', 0.8: 'red', 1: 'darkred'},
                    show=False
                ).add_to(mapa)
                print("✅ Mapa de calor por importe añadido")

        featuregroups_combo = {}
        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

        resumen_combos = {}
        resumen_mets = {}
        color_idx = 0
        met_general_idx = 0
        
        # Crear una capa por cada combinación MET-Ruta
        for met in mets_seleccionados:
            met_clean = str(met).strip()
            if met_clean.startswith('MET '):
                met_clean = met_clean[4:]
            met_id = met_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
            met_recorridos = df_recorridos[(df_recorridos[col_met] == met) & (df_recorridos[col_ruta].isin(rutas_seleccionadas))]
            rutas_met = met_recorridos[col_ruta].unique()
            
            for ruta in rutas_met:
                ruta_clean = str(ruta).strip()
                ruta_id = ruta_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
                combo_id = f"{met_id}-{ruta_id}"
                capa_nombre = f"MET {met_clean} - Ruta {ruta_clean}"
                fg_combo = folium.FeatureGroup(name=capa_nombre, show=True)
                featuregroups_combo[combo_id] = fg_combo
                ruta_recorridos = met_recorridos[met_recorridos[col_ruta] == ruta]
                
                if ruta_recorridos.empty:
                    continue
                    
                ruta_recorridos_sorted = ruta_recorridos.sort_values(col_secuencia)
                coords = list(zip(ruta_recorridos_sorted[col_latitud], ruta_recorridos_sorted[col_longitud]))
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                # Añadir línea de ruta solo si no es solo mapa de calor
                if tipo_visualizacion.value != 'solo_heatmap':
                    folium.PolyLine(
                        coords, 
                        color=color_ruta, 
                        weight=5, 
                        opacity=0.8, 
                        tooltip=capa_nombre,
                        className=f'ruta-line ruta-{combo_id}',
                        **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                    ).add_to(fg_combo)
                
                # Crear polígono del área de ruta (convex hull de todos los puntos)
                puntos_ruta = [Point(row[col_longitud], row[col_latitud]) for _, row in ruta_recorridos_sorted.iterrows()]
                if len(puntos_ruta) >= 3:
                    multipoint = MultiPoint(puntos_ruta)
                    hull = multipoint.convex_hull
                    if hull.geom_type == 'Polygon':
                        coords_polygon = [(lat, lon) for lon, lat in hull.exterior.coords]
                        area_popup = f'''
                        <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                            <div style="font-size: 14px; font-weight: bold;">📍 Área de Cobertura</div>
                            <div style="font-size: 12px; margin-top: 4px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</div>
                            <div style="font-size: 11px; margin-top: 2px;">👥 {ruta_recorridos[ruta_recorridos[col_tipo] == 'Cliente'].shape[0]} clientes</div>
                        </div>
                        '''
                        folium.Polygon(
                            locations=coords_polygon,
                            color=color_ruta,
                            weight=2,
                            opacity=0.7,
                            fillColor=color_ruta,
                            fillOpacity=0.15,
                            popup=folium.Popup(area_popup, max_width=250),
                            className=f'area-ruta area-{combo_id}',
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                
                for i, row in ruta_recorridos_sorted.iterrows():
                    # Marcadores para clientes (MEJORADOS)
                    if row[col_tipo] == 'Cliente' and tipo_visualizacion.value != 'solo_heatmap':
                        volumen = row[col_volumen] if col_volumen in df_recorridos.columns else 0
                        importe = row[col_importe] if col_importe in df_recorridos.columns else 0
                        tamano_marcador = obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min, tamano_max)
                        color_importe = obtener_color_importe(importe, imp_min, imp_max)
                        
                        # Calcular percentiles para mostrar en el popup
                        percentil_vol = ((volumen - vol_min) / (vol_max - vol_min) * 100) if vol_max > vol_min else 0
                        percentil_imp = ((importe - imp_min) / (imp_max - imp_min) * 100) if imp_max > imp_min else 0
                        
                        # Determinar categoría de cliente
                        if percentil_vol >= 80 and percentil_imp >= 80:
                            categoria = "⭐ VIP (Alto Volumen + Alto Importe)"
                            categoria_color = "#FFD700"
                        elif percentil_vol >= 70 or percentil_imp >= 70:
                            categoria = "🔸 Premium (Alto en una métrica)"
                            categoria_color = "#FF8C00"
                        elif percentil_vol >= 50 or percentil_imp >= 50:
                            categoria = "🔹 Estándar"
                            categoria_color = "#4682B4"
                        else:
                            categoria = "⚪ Básico"
                            categoria_color = "#808080"
                        
                        if tipo_visualizacion.value == 'marcadores_smart':
                            popup_html = f'''
                            <div style="background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%); border-radius: 16px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 16px; border: 3px solid {color_importe}; min-width: 320px; max-width: 400px; font-family: 'Segoe UI', Arial, sans-serif;">
                                <div style="background: {color_importe}; color: white; margin: -16px -16px 12px -16px; padding: 12px; border-radius: 13px 13px 8px 8px; text-align: center;">
                                    <div style="font-size:18px; font-weight:bold;">👨‍💼 {row[col_codigo]}</div>
                                    <div style="font-size:12px; opacity:0.9;">{row[col_nombre]}</div>
                                </div>
                                <div style="background: {categoria_color}; color: white; padding: 6px 10px; border-radius: 20px; text-align: center; margin-bottom: 12px; font-size: 12px; font-weight: bold;">"
                                    {categoria}
                                </div>
                                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 12px;">
                                    <div style="background: #E8F5E8; padding: 8px; border-radius: 8px; text-align: center;">
                                        <div style="font-size: 12px; color: #2E7D32;">📦 VOLUMEN</div>
                                        <div style="font-size: 16px; font-weight: bold; color: #1B5E20;">{volumen:,.3f}</div>
                                        <div style="font-size: 10px; color: #388E3C;">Top {100-percentil_vol:.0f}°</div>
                                        <div style="background: #C8E6C9; height: 4px; border-radius: 2px; margin-top: 4px;">
                                            <div style="background: #4CAF50; height: 100%; width: {percentil_vol:.0f}%; border-radius: 2px;"></div>
                                        </div>
                                    </div>
                                    <div style="background: #E3F2FD; padding: 8px; border-radius: 8px; text-align: center;">
                                        <div style="font-size: 12px; color: #1976D2;">💰 IMPORTE</div>
                                        <div style="font-size: 16px; font-weight: bold; color: #0D47A1;">${importe:,.0f}</div>
                                        <div style="font-size: 10px; color: #1976D2;">Top {100-percentil_imp:.0f}°</div>
                                        <div style="background: #BBDEFB; height: 4px; border-radius: 2px; margin-top: 4px;">
                                            <div style="background: {color_importe}; height: 100%; width: {percentil_imp:.0f}%; border-radius: 2px;"></div>
                                        </div>
                                    </div>
                                </div>
                                <div style="font-size: 11px; color: #666; text-align: center; border-top: 1px solid #ddd; padding-top: 8px;">
                                    🗺️ Ruta: <b>{row[col_ruta]}</b> | 🔢 Pos: <b>{row[col_secuencia]}</b> | 📏 Dist: <b>{row[col_distancia_sig]} km</b>
                                </div>
                            </div>
                            '''
                        else:
                            # Popup básico para marcadores normales
                            popup_html = f'''
                            <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_importe}; min-width: 280px;">
                                <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                                    👨‍💼 {row[col_codigo]} - {row[col_nombre]}
                                </div>
                                <div style="font-size:14px; color:#FF6B00; margin-bottom:4px;">
                                    📦 Volumen: {volumen:,.3f} (Top {100-percentil_vol:.0f}°)
                                </div>
                                <div style="font-size:14px; color:{color_importe}; margin-bottom:4px;">
                                    💰 Importe: ${importe:,.2f} (Top {100-percentil_imp:.0f}°)
                                </div>
                                <div style="font-size:12px; color:#666;">
                                    🗺️ {row[col_ruta]} | Pos: {row[col_secuencia]} | Dist: {row[col_distancia_sig]} km
                                </div>
                            </div>
                            '''
                        
                        # Marcador principal con tamaño por volumen y color por importe
                        folium.CircleMarker(
                            location=[row[col_latitud], row[col_longitud]],
                            popup=folium.Popup(popup_html, max_width=420),
                            radius=tamano_marcador,
                            color='#333333',
                            weight=3,
                            fillColor=color_importe,
                            fillOpacity=0.85,
                            className=f'cliente-marker cliente-{combo_id}',
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                        
                        # Marcador con número de secuencia (en capa separada para control)
                        folium.Marker(
                            location=[row[col_latitud], row[col_longitud]],
                            icon=DivIcon(
                                icon_size=(18,18),
                                icon_anchor=(9,9),
                                html=f'<div class="secuencia-marker secuencia-{combo_id}" style="font-size:10px; color:white; background:{color_ruta}; border-radius:50%; width:18px; height:18px; text-align:center; line-height:18px; border:2px solid #fff; font-weight:bold; box-shadow:0 2px 4px rgba(0,0,0,0.3);">{row[col_secuencia]}</div>'
                            ),
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                    
                    # Marcadores para METs (sin cambios)
                    elif row[col_tipo] == 'MET' and tipo_visualizacion.value != 'solo_heatmap':
                        popup_html = f'''
                        <div style="background: #fff; border-radius: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 14px; border: 2px solid #8BC34A; min-width: 260px; max-width: 340px;">
                            <div style="font-size:20px; color:{color_ruta}; font-weight:bold; margin-bottom:4px;">
                                <span style="vertical-align:middle;">🏠</span> {row[col_tipo]}: <span style="color:{color_ruta};">{row[col_codigo]}</span>
                            </div>
                            <div style="font-size:16px; color:#333; margin-bottom:4px;">
                                <span style="vertical-align:middle;">📚</span> <b>Nombre:</b> {row[col_nombre]}
                            </div>
                            <div style="font-size:15px; color:#2A81CB; margin-bottom:4px;">
                                <span style="vertical-align:middle;">🗺️</span> <b>Ruta:</b> {row[col_ruta]}
                            </div>
                            <div style="font-size:15px; color:#CB2A2A; margin-bottom:4px;">
                                <span style="vertical-align:middle;">↩️</span> <b>Distancia al siguiente:</b> {row[col_distancia_sig]} km
                            </div>
                            <div style="font-size:15px; color:#FFC107; margin-bottom:4px;">
                                <span style="vertical-align:middle;">🔢</span> <b>Número de secuencia:</b> {row[col_secuencia]}
                            </div>
                            <div style="font-size:14px; color:#333; margin-bottom:2px;">MET: <b>{met}</b></div>
                        </div>
                        '''
                        # Usar imagen personalizada para el MET si está disponible
                        if os.path.exists(icon_met_path):
                            icon_met = CustomIcon(
                                icon_image=icon_met_path,
                                icon_size=(40, 40),
                                icon_anchor=(20, 40),
                                popup_anchor=(0, -40)
                            )
                        else:
                            # Fallback al ícono por defecto si la imagen no existe
                            icon_met = folium.Icon(color=color_ruta, icon='home', prefix='fa')
                        
                        folium.Marker(
                            location=[row[col_latitud], row[col_longitud]],
                            popup=folium.Popup(popup_html, max_width=340),
                            icon=icon_met,
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                        
                        folium.Marker(
                            location=[row[col_latitud], row[col_longitud]],
                            icon=DivIcon(
                                icon_size=(24,24),
                                icon_anchor=(12,12),
                                html=f'<div style="font-size:16px; color:white; background:{color_ruta}; border-radius:50%; width:24px; height:24px; text-align:center; line-height:24px; border:2px solid #fff;">{row[col_secuencia]}</div>'
                            ),
                        ).add_to(fg_combo)
                
                # Agregar la capa al mapa
                fg_combo.add_to(mapa)
                
                # Calcular estadísticas para el resumen
                clientes_ruta = ruta_recorridos[ruta_recorridos[col_tipo] == 'Cliente']
                total_clientes_ruta = clientes_ruta.shape[0]
                
                # Buscar columna de distancia total de ruta
                columnas_distancia_total = [col for col in df_recorridos.columns if 'distancia' in col.lower() and 'total' in col.lower() and 'ruta' in col.lower()]
                if columnas_distancia_total:
                    distancia_total_ruta = ruta_recorridos[columnas_distancia_total[0]].dropna().unique()
                    distancia_total_ruta = distancia_total_ruta[0] if len(distancia_total_ruta) > 0 else 0
                else:
                    distancia_total_ruta = 0
                distancia_promedio_ruta = distancia_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                
                volumen_total_ruta = clientes_ruta[col_volumen].sum() if col_volumen in df_recorridos.columns else 0
                importe_total_ruta = clientes_ruta[col_importe].sum() if col_importe in df_recorridos.columns else 0
                volumen_promedio_ruta = volumen_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                importe_promedio_ruta = importe_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                
                resumen_combos[combo_id] = {
                    'met': met,
                    'ruta': ruta,
                    'clientes': total_clientes_ruta,
                    'distancia_total_km': distancia_total_ruta,
                    'distancia_promedio_cliente_km': distancia_promedio_ruta,
                    'volumen_total': volumen_total_ruta,
                    'volumen_promedio': volumen_promedio_ruta,
                    'importe_total': importe_total_ruta,
                    'importe_promedio': importe_promedio_ruta,
                    'color': color_ruta
                }

        # Agregar control de capas
        folium.LayerControl(collapsed=False, autoZIndex=True).add_to(mapa)

        # CSS para scroll en el control de capas
        scroll_layers_css = '''<style>.leaflet-control-layers-list { max-height: 220px; overflow-y: auto; overflow-x: hidden; width: 100%; min-width: 180px; }</style>'''
        mapa.get_root().html.add_child(folium.Element(scroll_layers_css))

        # Control personalizado para marcadores de secuencia y polígonos de área
        control_personalizado = '''
        <div id="controls-personalizados" style="position: fixed; top: 100px; left: 10px; z-index: 1000; background: white; padding: 10px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.3); font-family: 'Segoe UI', Arial, sans-serif; min-width: 200px;">
            <h4 style="margin: 0 0 10px 0; color: #333; font-size: 14px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">🎛️ Controles del Mapa</h4>
            
            <div style="margin-bottom: 8px;">
                <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                    <input type="checkbox" id="toggle-lineas" checked style="margin-right: 8px;">
                    <span style="color: #333;">🛣️ Líneas de Rutas</span>
                </label>
            </div>
            
            <div style="margin-bottom: 8px;">
                <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                    <input type="checkbox" id="toggle-secuencia" checked style="margin-right: 8px;">
                    <span style="color: #333;">🔢 Números de Secuencia</span>
                </label>
            </div>
            
            <div style="margin-bottom: 8px;">
                <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                    <input type="checkbox" id="toggle-areas" style="margin-right: 8px;">
                    <span style="color: #333;">📍 Áreas de Cobertura</span>
                </label>
            </div>
            
            <div style="margin-top: 10px; padding-top: 8px; border-top: 1px solid #eee; font-size: 10px; color: #666;">
                <div>💡 <b>Tip:</b> Usa estos controles para</div>
                <div>personalizar la visualización</div>
            </div>
        </div>
        '''
        mapa.get_root().html.add_child(folium.Element(control_personalizado))

        # JavaScript para controlar la visibilidad
        control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            // Control de elementos del mapa
            const toggleLineas = document.getElementById('toggle-lineas');
            const toggleSecuencia = document.getElementById('toggle-secuencia');
            const toggleAreas = document.getElementById('toggle-areas');
            
            // Función para mostrar/ocultar líneas de rutas
            function toggleRutaLines() {
                const rutaLines = document.querySelectorAll('.ruta-line');
                const isVisible = toggleLineas.checked;
                rutaLines.forEach(line => {
                    if (line.style) {
                        line.style.display = isVisible ? 'block' : 'none';
                    }
                    // También controlar elementos SVG path
                    if (line.tagName === 'path') {
                        line.style.display = isVisible ? 'block' : 'none';
                    }
                });
                
                // Controlar elementos polyline también
                const polylines = document.querySelectorAll('path[stroke]');
                polylines.forEach(polyline => {
                    // Solo ocultar si es parte de una ruta (tiene color y stroke-width)
                    if (polyline.getAttribute('stroke-width') === '5') {
                        polyline.style.display = isVisible ? 'block' : 'none';
                    }
                });
            }
            
            // Función para mostrar/ocultar marcadores de secuencia
            function toggleSecuenciaMarkers() {
                const secuenciaMarkers = document.querySelectorAll('.secuencia-marker');
                const isVisible = toggleSecuencia.checked;
                secuenciaMarkers.forEach(marker => {
                    marker.parentElement.style.display = isVisible ? 'block' : 'none';
                });
            }
            
            // Función para mostrar/ocultar áreas de cobertura
            function toggleAreaPolygons() {
                const areaPolygons = document.querySelectorAll('.area-ruta');
                const isVisible = toggleAreas.checked;
                areaPolygons.forEach(polygon => {
                    polygon.style.display = isVisible ? 'block' : 'none';
                });
            }
            
            // Event listeners
            toggleLineas.addEventListener('change', toggleRutaLines);
            toggleSecuencia.addEventListener('change', toggleSecuenciaMarkers);
            toggleAreas.addEventListener('change', toggleAreaPolygons);
            
            // Estado inicial - las áreas empiezan ocultas
            setTimeout(() => {
                toggleAreaPolygons();
                // Aplicar estado inicial a las líneas también
                toggleRutaLines();
            }, 1000);
            
            // Aplicar estilos CSS adicionales
            const style = document.createElement('style');
            style.textContent = `
                .ruta-line {
                    transition: opacity 0.3s ease;
                }
                .secuencia-marker {
                    transition: opacity 0.3s ease;
                }
                .area-ruta {
                    transition: opacity 0.3s ease;
                }
                .leaflet-div-icon {
                    background: transparent !important;
                    border: none !important;
                }
                path[stroke-width="5"] {
                    transition: opacity 0.3s ease;
                }
            `;
            document.head.appendChild(style);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(control_js))
        
        # Título del mapa MEJORADO
        desc_visual = {
            'marcadores_smart': '🎯 Marcadores Inteligentes: 📦 Tamaño = Volumen | 🎨 Color = Importe | ⭐ Categorías VIP',
            'marcadores': '📍 Marcadores: 📦 Tamaño = Volumen | 🎨 Color = Importe (🟡→🟠→🔴)',
            'solo_heatmap': '🔥 Mapas de Calor: Concentración de Volumen e Importe',
            'completo': '🎨 Vista Completa: 📍 Marcadores + 🔥 Mapas de Calor'
        }
        
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
            <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                🗺️ ANÁLISIS COMERCIAL - VOLUMEN E IMPORTE
            </h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas: <b>{len(rutas_seleccionadas)}</b> | 👥 Clientes: <b>{len(clientes_todos)}</b><br>
                {desc_visual.get(tipo_visualizacion.value, '')}
            </p>
        </div>
        '''
        mapa.get_root().html.add_child(folium.Element(titulo_html))

        # Panel de resumen (simplificado para evitar sobrecarga)
        if len(resumen_combos) <= 5:  # Solo mostrar resumen si hay pocas rutas
            resumen_html = f'''
            <div id="resumen-general" style="position: fixed; top: 20px; right: 20px; width: 300px; max-height: 400px; overflow-y: auto; background: rgba(255,255,255,0.95); border: 2px solid #333; z-index: 9997; font-size: 11px; padding: 12px; border-radius: 12px; box-shadow: 0 4px 16px rgba(0,0,0,0.2);">
                <h3 style="margin: 0 0 10px 0; color: #333; text-align: center; font-size: 14px;">📊 Resumen de Análisis</h3>
            '''
            
            for combo_id, resumen in resumen_combos.items():
                resumen_html += f'''
                <div style="margin-bottom: 8px; padding: 8px; background: #f8f9fa; border-radius: 6px; border-left: 4px solid {resumen['color']};">
                    <div style="font-weight: bold; color: {resumen['color']}; font-size: 12px;">
                        🏠 {resumen['met']} - 🛣️ {resumen['ruta']}
                    </div>
                    <div style="font-size: 10px; color: #666; margin-top: 2px;">
                        👥 {resumen['clientes']} clientes | 📦 Vol: {resumen['volumen_total']:,.0f} | 💰 ${resumen['importe_total']:,.0f}
                    </div>
                </div>
                '''
            
            resumen_html += "</div>"
            mapa.get_root().html.add_child(folium.Element(resumen_html))

        print(f"✅ Mapa generado exitosamente con {len(featuregroups_combo)} capas")
        
        # Guardar el mapa
        tipo_viz = tipo_visualizacion.value.replace('_', '-')
        nombre_mapa = os.path.join(output_dir, f'mapa_analisis_comercial_{tipo_viz}_{fecha_hora}.html')
        try:
            mapa.save(nombre_mapa)
            print(f'🗺️ Mapa de análisis comercial exportado a: {nombre_mapa}')
            print(f"🎯 Visualización utilizada: {desc_visual.get(tipo_visualizacion.value, tipo_visualizacion.value)}")
        except Exception as e:
            print(f'❌ Error al guardar el mapa: {e}')

# Conectar el botón con la función
generar_btn.on_click(generar_mapa)

def generar_analisis_excel(b):
    with output_excel:
        clear_output()
        
        # Verificar que los datos estén disponibles
        if df_recorridos is None or df_recorridos.empty:
            print('❌ No hay datos válidos para generar el análisis.')
            return
            
        mets_seleccionados = list(met_selector.value)
        rutas_seleccionadas = list(ruta_selector.value)
        if not mets_seleccionados:
            print('Selecciona al menos un MET para generar el análisis.')
            return
        if not rutas_seleccionadas:
            print('Selecciona al menos una ruta para generar el análisis.')
            return

        print(f"📊 Generando análisis Excel para {len(mets_seleccionados)} METs y {len(rutas_seleccionadas)} rutas...")
        
        # Filtrar datos según selección
        datos_filtrados = df_recorridos[
            (df_recorridos[col_met].isin(mets_seleccionados)) & 
            (df_recorridos[col_ruta].isin(rutas_seleccionadas))
        ]
        
        clientes_filtrados = datos_filtrados[datos_filtrados[col_tipo] == 'Cliente'].copy()
        
        if clientes_filtrados.empty:
            print("❌ No se encontraron clientes en la selección actual")
            return
        
        print(f"👥 Analizando {len(clientes_filtrados)} clientes...")
        
        # Calcular percentiles y rankings
        clientes_filtrados['percentil_volumen'] = clientes_filtrados[col_volumen].rank(pct=True) * 100
        clientes_filtrados['percentil_importe'] = clientes_filtrados[col_importe].rank(pct=True) * 100
        clientes_filtrados['ranking_volumen'] = clientes_filtrados[col_volumen].rank(ascending=False, method='dense').astype(int)
        clientes_filtrados['ranking_importe'] = clientes_filtrados[col_importe].rank(ascending=False, method='dense').astype(int)
        
        # Calcular categorías de cliente
        def calcular_categoria(row):
            if row['percentil_volumen'] >= 80 and row['percentil_importe'] >= 80:
                return "VIP"
            elif row['percentil_volumen'] >= 70 or row['percentil_importe'] >= 70:
                return "Premium"
            elif row['percentil_volumen'] >= 50 or row['percentil_importe'] >= 50:
                return "Estándar"
            else:
                return "Básico"
        
        clientes_filtrados['categoria'] = clientes_filtrados.apply(calcular_categoria, axis=1)
        
        # Crear análisis por MET
        resumen_mets = []
        for met in mets_seleccionados:
            clientes_met = clientes_filtrados[clientes_filtrados[col_met] == met]
            if not clientes_met.empty:
                resumen = {
                    'MET': met,
                    'Total_Clientes': len(clientes_met),
                    'Volumen_Total': clientes_met[col_volumen].sum(),
                    'Volumen_Promedio': clientes_met[col_volumen].mean(),
                    'Volumen_Mediana': clientes_met[col_volumen].median(),
                    'Importe_Total': clientes_met[col_importe].sum(),
                    'Importe_Promedio': clientes_met[col_importe].mean(),
                    'Importe_Mediana': clientes_met[col_importe].median(),
                    'Distancia_Total': clientes_met[col_distancia_sig].sum(),
                    'Distancia_Promedio': clientes_met[col_distancia_sig].mean(),
                    'Clientes_VIP': len(clientes_met[clientes_met['categoria'] == 'VIP']),
                    'Clientes_Premium': len(clientes_met[clientes_met['categoria'] == 'Premium']),
                    'Eficiencia_Volumen_Distancia': clientes_met[col_volumen].sum() / max(clientes_met[col_distancia_sig].sum(), 1),
                    'Eficiencia_Importe_Distancia': clientes_met[col_importe].sum() / max(clientes_met[col_distancia_sig].sum(), 1),
                    'Score_Rentabilidad': (clientes_met[col_volumen].sum() * 0.4 + clientes_met[col_importe].sum() * 0.6) / max(clientes_met[col_distancia_sig].sum(), 1)
                }
                resumen_mets.append(resumen)
        
        df_resumen_mets = pd.DataFrame(resumen_mets)
        
        # Ranking de METs por diferentes criterios
        df_resumen_mets['Ranking_Volumen_Total'] = df_resumen_mets['Volumen_Total'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Importe_Total'] = df_resumen_mets['Importe_Total'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Eficiencia_Vol'] = df_resumen_mets['Eficiencia_Volumen_Distancia'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Eficiencia_Imp'] = df_resumen_mets['Eficiencia_Importe_Distancia'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Rentabilidad'] = df_resumen_mets['Score_Rentabilidad'].rank(ascending=False, method='dense').astype(int)
        
        # Crear archivo Excel
        nombre_excel = os.path.join(output_dir, f'analisis_comercial_mets_{fecha_hora}.xlsx')
        wb = Workbook()
        
        # Configurar estilos
        header_font = Font(bold=True, color="FFFFFF")
        header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
        vip_fill = PatternFill(start_color="FFD700", end_color="FFD700", fill_type="solid")
        premium_fill = PatternFill(start_color="FF8C00", end_color="FF8C00", fill_type="solid")
        estandar_fill = PatternFill(start_color="4682B4", end_color="4682B4", fill_type="solid")
        basico_fill = PatternFill(start_color="808080", end_color="808080", fill_type="solid")
        
        # 1. Hoja: Resumen Ejecutivo de METs
        ws_resumen = wb.active
        ws_resumen.title = "Resumen METs"
        
        # Escribir datos del resumen
        for r in dataframe_to_rows(df_resumen_mets, index=False, header=True):
            ws_resumen.append(r)
        
        # Aplicar formato a encabezados
        for cell in ws_resumen[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center")
        
        # Ajustar ancho de columnas
        for column in ws_resumen.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            ws_resumen.column_dimensions[column_letter].width = min(max_length + 2, 20)
        
        # 2. Hoja: Top Clientes por Volumen
        ws_top_vol = wb.create_sheet("Top Clientes Volumen")
        top_clientes_vol = clientes_filtrados.nlargest(50, col_volumen)[
            [col_met, col_ruta, col_codigo, col_nombre, col_volumen, 'percentil_volumen', 
             'ranking_volumen', col_distancia_sig, 'categoria']
        ].copy()
        
        for r in dataframe_to_rows(top_clientes_vol, index=False, header=True):
            ws_top_vol.append(r)
        
        # Aplicar formato
        for cell in ws_top_vol[1]:
            cell.font = header_font
            cell.fill = header_fill
        
        # Aplicar colores por categoría
        for row in range(2, len(top_clientes_vol) + 2):
            categoria = ws_top_vol[f'I{row}'].value
            fill = None
            if categoria == 'VIP':
                fill = vip_fill
            elif categoria == 'Premium':
                fill = premium_fill
            elif categoria == 'Estándar':
                fill = estandar_fill
            elif categoria == 'Básico':
                fill = basico_fill
            
            if fill:
                for col in range(1, 10):
                    ws_top_vol.cell(row=row, column=col).fill = fill
        
        # 3. Hoja: Top Clientes por Importe
        ws_top_imp = wb.create_sheet("Top Clientes Importe")
        top_clientes_imp = clientes_filtrados.nlargest(50, col_importe)[
            [col_met, col_ruta, col_codigo, col_nombre, col_importe, 'percentil_importe', 
             'ranking_importe', col_distancia_sig, 'categoria']
        ].copy()
        
        for r in dataframe_to_rows(top_clientes_imp, index=False, header=True):
            ws_top_imp.append(r)
        
        # Aplicar formato
        for cell in ws_top_imp[1]:
            cell.font = header_font
            cell.fill = header_fill
        
        # 4. Hoja: Análisis Detallado por Cliente
        ws_detalle = wb.create_sheet("Detalle Clientes")
        detalle_clientes = clientes_filtrados[
            [col_met, col_ruta, col_codigo, col_nombre, col_volumen, col_importe, 
             col_distancia_sig, 'percentil_volumen', 'percentil_importe', 
             'ranking_volumen', 'ranking_importe', 'categoria']
        ].copy()
        
        for r in dataframe_to_rows(detalle_clientes, index=False, header=True):
            ws_detalle.append(r)
        
        # Aplicar formato a encabezados
        for cell in ws_detalle[1]:
            cell.font = header_font
            cell.fill = header_fill
        
        # 5. Crear gráficos de dispersión
        ws_graficos = wb.create_sheet("Gráficos Dispersión")
        
        # Preparar datos para gráficos por MET
        row_actual = 1
        for met in mets_seleccionados:
            clientes_met = clientes_filtrados[clientes_filtrados[col_met] == met]
            if len(clientes_met) < 2:
                continue
                
            # Título
            ws_graficos[f'A{row_actual}'] = f"MET: {met}"
            ws_graficos[f'A{row_actual}'].font = Font(bold=True, size=14)
            row_actual += 2
            
            # Datos para gráfico de volumen vs distancia
            ws_graficos[f'A{row_actual}'] = "Distancia (km)"
            ws_graficos[f'B{row_actual}'] = "Volumen"
            ws_graficos[f'A{row_actual}'].font = header_font
            ws_graficos[f'B{row_actual}'].font = header_font
            row_actual += 1
            
            start_row_vol = row_actual
            for _, cliente in clientes_met.iterrows():
                ws_graficos[f'A{row_actual}'] = cliente[col_distancia_sig]
                ws_graficos[f'B{row_actual}'] = cliente[col_volumen]
                row_actual += 1
            
            # Crear gráfico de dispersión para volumen
            chart_vol = ScatterChart()
            chart_vol.title = f"Volumen vs Distancia - {met}"
            chart_vol.style = 2
            chart_vol.x_axis.title = "Distancia (km)"
            chart_vol.y_axis.title = "Volumen"
            
            xvalues = Reference(ws_graficos, min_col=1, min_row=start_row_vol, max_row=row_actual-1)
            yvalues = Reference(ws_graficos, min_col=2, min_row=start_row_vol, max_row=row_actual-1)
            series = Series(yvalues, xvalues, title="Volumen")
            chart_vol.series.append(series)
            
            ws_graficos.add_chart(chart_vol, f"D{start_row_vol-2}")
            
            row_actual += 2
            
            # Datos para gráfico de importe vs distancia
            ws_graficos[f'A{row_actual}'] = "Distancia (km)"
            ws_graficos[f'B{row_actual}'] = "Importe"
            ws_graficos[f'A{row_actual}'].font = header_font
            ws_graficos[f'B{row_actual}'].font = header_font
            row_actual += 1
            
            start_row_imp = row_actual
            for _, cliente in clientes_met.iterrows():
                ws_graficos[f'A{row_actual}'] = cliente[col_distancia_sig]
                ws_graficos[f'B{row_actual}'] = cliente[col_importe]
                row_actual += 1
            
            # Crear gráfico de dispersión para importe
            chart_imp = ScatterChart()
            chart_imp.title = f"Importe vs Distancia - {met}"
            chart_imp.style = 3
            chart_imp.x_axis.title = "Distancia (km)"
            chart_imp.y_axis.title = "Importe ($)"
            
            xvalues_imp = Reference(ws_graficos, min_col=1, min_row=start_row_imp, max_row=row_actual-1)
            yvalues_imp = Reference(ws_graficos, min_col=2, min_row=start_row_imp, max_row=row_actual-1)
            series_imp = Series(yvalues_imp, xvalues_imp, title="Importe")
            chart_imp.series.append(series_imp)
            
            ws_graficos.add_chart(chart_imp, f"L{start_row_imp-2}")
            
            row_actual += 4
        
        # 6. Hoja: Recomendaciones
        ws_recomendaciones = wb.create_sheet("Recomendaciones")
        
        # Calcular recomendaciones
        mejor_met_volumen = df_resumen_mets.loc[df_resumen_mets['Volumen_Total'].idxmax()]
        mejor_met_importe = df_resumen_mets.loc[df_resumen_mets['Importe_Total'].idxmax()]
        mejor_met_eficiencia = df_resumen_mets.loc[df_resumen_mets['Score_Rentabilidad'].idxmax()]
        
        recomendaciones = [
            ["ANÁLISIS DE CONVENIENCIA POR MET", ""],
            ["", ""],
            ["🏆 MEJOR MET POR VOLUMEN TOTAL:", mejor_met_volumen['MET']],
            ["📦 Volumen Total:", f"{mejor_met_volumen['Volumen_Total']:,.2f}"],
            ["👥 Total Clientes:", f"{mejor_met_volumen['Total_Clientes']:,}"],
            ["", ""],
            ["🏆 MEJOR MET POR IMPORTE TOTAL:", mejor_met_importe['MET']],
            ["💰 Importe Total:", f"${mejor_met_importe['Importe_Total']:,.2f}"],
            ["👥 Total Clientes:", f"{mejor_met_importe['Total_Clientes']:,}"],
            ["", ""],
            ["🏆 MEJOR MET POR RENTABILIDAD:", mejor_met_eficiencia['MET']],
            ["📊 Score Rentabilidad:", f"{mejor_met_eficiencia['Score_Rentabilidad']:.2f}"],
            ["⚡ Eficiencia Vol/Dist:", f"{mejor_met_eficiencia['Eficiencia_Volumen_Distancia']:.2f}"],
            ["⚡ Eficiencia Imp/Dist:", f"{mejor_met_eficiencia['Eficiencia_Importe_Distancia']:.2f}"],
            ["", ""],
            ["CRITERIOS DE EVALUACIÓN:", ""],
            ["• Volumen Total: Cantidad total de productos", ""],
            ["• Importe Total: Valor económico total", ""],
            ["• Score Rentabilidad: (Vol*0.4 + Imp*0.6)/Distancia", ""],
            ["• Eficiencia: Ratio valor/distancia recorrida", ""],
        ]
        
        for row_data in recomendaciones:
            ws_recomendaciones.append(row_data)
        
        # Aplicar formato a recomendaciones
        ws_recomendaciones['A1'].font = Font(bold=True, size=16)
        for row in [3, 7, 11]:
            ws_recomendaciones[f'A{row}'].font = Font(bold=True, color="FF0000")
            ws_recomendaciones[f'B{row}'].font = Font(bold=True, color="FF0000")
        
        # Guardar archivo
        try:
            wb.save(nombre_excel)
            print(f"✅ Análisis Excel generado exitosamente: {nombre_excel}")
            print(f"📋 Hojas incluidas:")
            print(f"   • Resumen METs - Comparativa general")
            print(f"   • Top Clientes Volumen - Mejores 50 por volumen")
            print(f"   • Top Clientes Importe - Mejores 50 por importe")
            print(f"   • Detalle Clientes - Análisis completo de todos los clientes")
            print(f"   • Gráficos Dispersión - Volumen e Importe vs Distancia por MET")
            print(f"   • Recomendaciones - Cuál MET conviene más según diferentes criterios")
            print(f"")
            print(f"🏆 RESUMEN EJECUTIVO:")
            print(f"   📦 Mejor MET por Volumen: {mejor_met_volumen['MET']}")
            print(f"   💰 Mejor MET por Importe: {mejor_met_importe['MET']}")
            print(f"   ⚡ Mejor MET por Rentabilidad: {mejor_met_eficiencia['MET']}")
            
        except Exception as e:
            print(f"❌ Error al guardar el archivo Excel: {e}")

# Conectar el nuevo botón
analisis_excel_btn.on_click(generar_analisis_excel)

# Interfaz de usuario MEJORADA
display(widgets.VBox([
    widgets.HTML('<h2 style="color: #2E7D32; text-align: center; margin-bottom: 20px;">🗺️ Generador de Mapas de Análisis Comercial</h2>'),
    
    widgets.HTML('<h3 style="color: #1976D2;">📍 Selección de Ubicaciones</h3>'),
    widgets.HBox([met_selector, ruta_selector]),
    widgets.HBox([todas_rutas_checkbox]),
    
    widgets.HTML('<h3 style="color: #7B1FA2;">🎨 Opciones de Visualización</h3>'),
    tipo_visualizacion,
    widgets.HBox([escala_marcadores]),
    
    widgets.HTML('<h3 style="color: #F57C00;">⚙️ Configuración</h3>'),
    widgets.HBox([clientes_a_procesar, todos_clientes_checkbox]),
    
    widgets.HTML('''
    <div style="background: #E8F5E8; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #4CAF50;">
        <h4 style="margin: 0 0 8px 0; color: #2E7D32;">📋 Guía de Visualización:</h4>
        <ul style="margin: 0; padding-left: 20px; font-size: 12px; color: #388E3C;">
            <li><b>🎯 Marcadores Inteligentes:</b> Categorización automática de clientes (VIP, Premium, Estándar, Básico)</li>
            <li><b>📦 Tamaño del marcador:</b> Proporcional al volumen (más grande = mayor volumen)</li>
            <li><b>🎨 Color del marcador:</b> Representa el importe (🟡 Bajo → 🟠 Medio → 🔴 Alto)</li>
            <li><b>🔥 Mapa de calor:</b> Muestra concentración geográfica de valores</li>
            <li><b>⭐ Categorías:</b> VIP (Top 20%), Premium (Top 30%), Estándar (Top 50%), Básico (resto)</li>
            <li><b>🎯 Percentiles dinámicos:</b> Se calculan basándose en METs y rutas seleccionados</li>
        </ul>
    </div>
    '''),
    
    widgets.HTML('<h3 style="color: #E65100;">🚀 Generar Análisis</h3>'),
    widgets.HBox([generar_btn, analisis_excel_btn]),
    
    widgets.HTML('''
    <div style="background: #E3F2FD; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #2196F3;">
        <h4 style="margin: 0 0 8px 0; color: #1565C0;">📊 Análisis Excel Incluye:</h4>
        <ul style="margin: 0; padding-left: 20px; font-size: 12px; color: #1976D2;">
            <li><b>🏆 Rankings y Top Clientes:</b> Por volumen e importe con percentiles</li>
            <li><b>📈 Gráficos de Dispersión:</b> Volumen vs Distancia e Importe vs Distancia por MET</li>
            <li><b>⚡ Análisis de Eficiencia:</b> Ratios de rentabilidad por MET</li>
            <li><b>🎯 Recomendaciones:</b> Cuál MET conviene más según diferentes criterios</li>
            <li><b>📋 Categorización:</b> Clientes VIP, Premium, Estándar y Básicos</li>
            <li><b>📊 Métricas Comparativas:</b> Estadísticas completas por MET</li>
        </ul>
    </div>
    '''),
    
    output_map,
    output_excel
], layout=widgets.Layout(padding='20px')))

Archivo seleccionado: celaya1_celya2_bajio1.xlsx
📋 Hojas disponibles en el archivo:
   1. Bsae
✅ Hoja seleccionada automáticamente: 'Bsae'
📊 Datos cargados: 13368 filas, 11 columnas
🔍 Columnas encontradas:
   1. MET
   2. Ruta
   3. Secuencia
   4. Tipo
   5. Codigo
   6. Longitud
   7. Latitud
   8. Nombre
   9. Distancia al siguiente
   10. Volumen promedio diario
   ... y 1 columnas más
🔍 Mapeo automático de columnas:
   MET: MET
   Ruta: Ruta
   Secuencia: Secuencia
   Tipo: Tipo
   Codigo: Codigo
   Longitud: Longitud
   Latitud: Latitud
   Nombre: Nombre
🔍 Verificando columnas requeridas:
   ✅ MET
   ✅ Ruta
   ✅ Secuencia
   ✅ Tipo
   ✅ Codigo
   ✅ Longitud
   ✅ Latitud
   ✅ Nombre
🔍 Verificando columnas opcionales:
   ✅ Distancia al siguiente
   ✅ Volumen promedio diario
   ✅ Importe promedio diario

✅ Todas las columnas críticas están disponibles
✅ Columna Volumen promedio diario normalizada
✅ Columna Importe promedio diario normalizada
📍 METs encontrados: 3
🛣️ Rutas encontrada